In [1]:
## Use google/gemma-4-12B-it
## to create results for text based problems

from google.colab import userdata
from huggingface_hub import login
import json
import time
import requests
import torch
from transformers import AutoProcessor, AutoModelForMultimodalLM

In [2]:
## check if cuda is available and GPU

print("CUDA available:", torch.cuda.is_available())
print("Using GPU:", torch.cuda.get_device_name(0))

CUDA available: True
Using GPU: NVIDIA A100-SXM4-40GB


In [3]:
## authenticate with huggingface

hf_token = userdata.get("hf_token")
login(token=hf_token)

In [4]:
model_name ="gemma-4"
model_id = "google/gemma-4-12B-it"


# Load model
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForMultimodalLM.from_pretrained(
    model_id,
    dtype="auto",
    device_map="auto"
)

processor_config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.42k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.09k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B / 23.9GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/677 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/260 [00:00<?, ?B/s]

In [5]:
## import the prompts from the text_prompts.py

import sys

sys.path.append("/content/drive/MyDrive/colab_files/Problems/Text")
from text_prompts import prompts

print(prompts)

{'regular_system_prompt': 'You are a physics expert. Based on the question that is given solve the problem. The question has equations and symbols in latex, but your answer should not be in latex.', 'detail_system_prompt': 'You are a physics expert. Based on the question that is given solve the problem, but if there are any physical laws broken then mention it and also say if the problem can be solved or not. The question has equations and symbols in latex, but your answer should not be in latex.'}


In [6]:
print(prompts.keys())

dict_keys(['regular_system_prompt', 'detail_system_prompt'])


In [7]:
from pathlib import Path

text_folder = Path("/content/drive/MyDrive/colab_files/Problems/Text")
file_names = sorted(file.name for file in text_folder.glob("*.md"))
print("Total number of files:::",len(file_names))
print(file_names)

Total number of files::: 24
['Problem_1_Obvious_Q.md', 'Problem_1_Regular_Q.md', 'Problem_1_non_obvious_Q.md', 'Problem_2_Obvious_Q.md', 'Problem_2_Regular_Q.md', 'Problem_2_non_obvious_Q.md', 'Problem_3_Obvious_Q.md', 'Problem_3_Regular_Q.md', 'Problem_3_non_obvious_Q.md', 'Problem_4_Obvious_Q.md', 'Problem_4_Regular_Q.md', 'Problem_4_non_obvious_Q.md', 'Problem_5_Obvious_Q.md', 'Problem_5_Regular_Q.md', 'Problem_5_non_obvious_Q.md', 'Problem_6_Obvious_Q.md', 'Problem_6_Regular_Q.md', 'Problem_6_non_obvious_Q.md', 'Problem_7_Obvious_Q.md', 'Problem_7_Regular_Q.md', 'Problem_7_non_obvious_Q.md', 'Problem_8_Obvious_Q.md', 'Problem_8_Regular_Q.md', 'Problem_8_non_obvious_Q.md']


In [8]:

## to calculate the total time taken to generate the output

def synchronize_cuda():
    # GPU operations are asynchronous.
    # Wait for pending operations before taking timing boundaries.
    if torch.cuda.is_available():
        for device_index in range(torch.cuda.device_count()):
            torch.cuda.synchronize(device_index)

In [9]:
%%time

for keys in prompts:
  for file in file_names:

    text_folder = Path("/content/drive/MyDrive/colab_files/Problems/Text") ## read the files
    file_path = text_folder/ file

    if keys == "detail_system_prompt" and "regular" in file_path.stem.lower():  ## skip if keys have detail_system_prompt and
      continue     ## problems that have regular in their name. We dont need Regular problems to use detail prompts but run the rest

    problem_text = file_path.read_text(encoding="utf-8")


    messages = [
    {
        "role": "system",
        "content": prompts[keys],
    },
    {
        "role": "user",
        "content": problem_text,
    },
      ]


    # Format and tokenize the conversation
    inputs = processor.apply_chat_template(
      messages,
      add_generation_prompt=True,
      tokenize=True,
      return_dict=True,
      return_tensors="pt",
          ).to(model.device)

    input_length = inputs["input_ids"].shape[-1]

    ## generate and measure time
    synchronize_cuda()
    start_time = time.perf_counter()

    # Generate the response
    with torch.inference_mode():
        outputs = model.generate(
          **inputs,
         max_new_tokens=10000,
          do_sample=False,
     )

    synchronize_cuda()
    end_time = time.perf_counter()

    ## calculate time taken
    total_time_seconds = end_time - start_time

    # Decode only the newly generated tokens
    generated_tokens = outputs[0, input_length:]
    output_tokens = generated_tokens.numel()

    response = processor.decode(
      generated_tokens,
      skip_special_tokens=True,
      clean_up_tokenization_spaces=False,
          )

    output_text = processor.parse_response(
    response,
    prefix=inputs["input_ids"],
        )
    output_filename= f"{model_name}_{file_path.stem}_{keys}" +".md"

    output_folder = Path("/content/drive/MyDrive/colab_files/Results/gemma-4_text")
    output_file_path = output_folder / output_filename

    # Extract string content from the parsed response dictionary if applicable
    output_content = output_text["content"] if isinstance(output_text, dict) and "content" in output_text else response
    output_file_path.write_text(output_content, encoding="utf-8")

    ## Build metadata

    metadata = {
    "requested_model": model_id,
    "problem_text": problem_text,
    "prompt_name": keys,
    "total_output_tokens": output_tokens,
    "total_time_taken": total_time_seconds
    }

    metadata_folder= Path("/content/drive/MyDrive/colab_files/Results/gemma-4_text/metadata")
    metadata_name= f"{model_name}_{file_path.stem}_{keys}" +".json"

    meta_data_file_path = metadata_folder / metadata_name
    meta_data_file_path.write_text(
                        json.dumps(metadata, indent=4, ensure_ascii=False),
                        encoding="utf-8",
                    )

    print(f"Completed:::{model_name}_{file_path.stem}_{keys}")

Completed:::gemma-4_Problem_1_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_1_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_1_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_2_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_2_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_2_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_3_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_3_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_3_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_4_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_4_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_4_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_5_Obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_5_Regular_Q_regular_system_prompt
Completed:::gemma-4_Problem_5_non_obvious_Q_regular_system_prompt
Completed:::gemma-4_Problem_6_Obvious_Q_regular_sy